# Comparison Table -- Classification Accuracy + Structural Bucket Breakdown

Cleaner copy of `RQ_4_5/table.ipynb`. The original notebook was a scratchpad with several
superseded drafts of the same scoring logic; this copy keeps only the final, necessary
computation:

1. **Classification accuracy** per output file, split into Overall / ADHD / Control accuracy.
2. **Structural bucket breakdown** for the hub- and community-identification tasks -- each
   response is graded into `correct (>= 0.6)` / `partial (0.4-0.59)` / `suboptimal (< 0.4)` /
   `hallucinatory` / `missing` against ground truth, per model x task x condition.

## 1. Classification accuracy -- Overall / ADHD / Control, per model x file

In [2]:
import os
import json
import pandas as pd

RQ45_DIR = "/home/siu856622573/Code/RQ_4_5"
VALID_LABELS = {"ADHD", "Control"}


def calculate_accuracy(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    # Only scorable subjects: valid ground truth AND valid prediction
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS
        and r.get("prediction") in VALID_LABELS
    ]

    overall_correct = sum(r["prediction"] == r["ground_truth"] for r in scorable)
    overall_accuracy = overall_correct / len(scorable) if scorable else None

    adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    adhd_correct = sum(r["prediction"] == "ADHD" for r in adhd)
    adhd_accuracy = adhd_correct / len(adhd) if adhd else None

    control = [r for r in scorable if r["ground_truth"] == "Control"]
    control_correct = sum(r["prediction"] == "Control" for r in control)
    control_accuracy = control_correct / len(control) if control else None

    return {
        "Overall Accuracy": overall_accuracy,
        "ADHD Accuracy": adhd_accuracy,
        "Control Accuracy": control_accuracy,
    }


def process_folder(folder_name):
    """Scans RQ45_DIR/<model>/<folder_name>/*.json for every model directory."""
    rows = []
    model_folders = sorted(
        name for name in os.listdir(RQ45_DIR)
        if os.path.isdir(os.path.join(RQ45_DIR, name))
    )

    for model in model_folders:
        target_dir = os.path.join(RQ45_DIR, model, folder_name)
        if not os.path.isdir(target_dir):
            continue

        json_files = sorted(f for f in os.listdir(target_dir) if f.endswith(".json"))
        for json_file in json_files:
            json_path = os.path.join(target_dir, json_file)
            try:
                scores = calculate_accuracy(json_path)
                rows.append({
                    "Model": model,
                    "File": json_file,
                    "Overall Accuracy": scores["Overall Accuracy"],
                    "ADHD Accuracy": scores["ADHD Accuracy"],
                    "Control Accuracy": scores["Control Accuracy"],
                })
            except Exception as e:
                print(f"Error processing: {json_path}\n  {e}")

    return pd.DataFrame(rows)


community_table = process_folder("community")
hub_table = process_folder("hub")

accuracy_columns = ["Overall Accuracy", "ADHD Accuracy", "Control Accuracy"]
for df in [community_table, hub_table]:
    for col in accuracy_columns:
        df[col] = df[col].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

print("\n" + "=" * 120)
print("COMMUNITY ACCURACY")
print("=" * 120)
print(community_table.to_string(index=False))

print("\n" + "=" * 120)
print("HUB ACCURACY")
print("=" * 120)
print(hub_table.to_string(index=False))



COMMUNITY ACCURACY
      Model                                                                File Overall Accuracy ADHD Accuracy Control Accuracy
   deepseek                    deepseek_community_for_classification_10com.json           62.11%         3.93%           95.49%
   deepseek            deepseek_community_for_classification_no_edge_10com.json           53.00%        41.73%           59.43%
   deepseek     deepseek_community_for_classification_no_edge_weight_10com.json           61.85%         3.57%           95.29%
   deepseek                                 deepseek_community_gt_36_10com.json           51.10%        41.43%           56.71%
   deepseek                               deepseek_community_gt_36_10com_1.json           51.39%        49.82%           52.28%
deepseek_v4                 deepseek_v4_community_for_classification_10com.json           62.84%         2.50%           97.54%
deepseek_v4         deepseek_v4_community_for_classification_no_edge_10com.json     

## 2. Structural bucket breakdown -- hub & community identification

Grades every prediction against ground truth into one of 5 buckets:
`correct (>= 0.6)` / `partial (0.4-0.59)` / `suboptimal (< 0.4)` / `hallucinatory` / `missing`.

Includes the pair-order-normalization fix (`normalize_pair`) for the community task: ground
truth stores between-network pairs in canonical `network_names` order, but the LLM's free-text
output isn't guaranteed to follow that order (e.g. `"Cont-DorsAttn"` vs. `"DorsAttn-Cont"`) --
without normalizing first, those get misgraded as hallucinatory instead of scored on overlap.

In [10]:
import glob


In [11]:
import os
import re
import json
import glob
import pandas as pd


# ═══════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════

RQ45_DIR = "/home/siu856622573/Code/RQ_4_5"
CODE_DIR = "/home/siu856622573/Code"

ROI_NAMES_JSON = os.path.join(
    CODE_DIR,
    "aal_roi_names.json"
)

EDGE_LIST_NAMES_ONLY = os.path.join(
    CODE_DIR,
    "edge_only_from_top10pos_names.json"
)

NETWORK_NAMES_JSON = os.path.join(
    CODE_DIR,
    "network_names.json"
)

HUB_GROUND_TRUTH = os.path.join(
    CODE_DIR,
    "hub_ground_truth_20roi.json"
)

COMMUNITY_GROUND_TRUTH_COUNTS = os.path.join(
    CODE_DIR,
    "community_ground_truth_counts.json"
)

TOP_K_GROUND_TRUTH = 10

OUTPUT_CSV = os.path.join(
    RQ45_DIR,
    "bucket_breakdown.csv"
)


# ═══════════════════════════════════════════════════════════════════
# EXACT FILES TO PROCESS
# ═══════════════════════════════════════════════════════════════════

TARGET_FILES = {
    "deepseek_hub_for_classification_no_edge_20hub.json",

    "deepseek_v4_hub_for_classification_20hub.json",

    "gpt_hub_for_classification_20hub.json",

    "llama_hub_for_classification_20hub_del.json",

    "qwen2.5_hub_for_classification_20hub.json",

    "qwen3_hub_for_classification_20hub.json",

    "deepseek_community_for_classification_no_edge_10com.json",

    "deepseek_v4_community_for_classification_10com.json",

    "gpt_community_for_classification_no_edge_10com.json",

    "llama_community_for_classification_10com.json",

    "qwen2.5_community_for_classification_10com.json",

    "qwen3_community_for_classification_10com.json",
}


# ═══════════════════════════════════════════════════════════════════
# BUCKETS
# ═══════════════════════════════════════════════════════════════════

BUCKETS = [
    "correct (>= 0.6)",
    "partial (0.4-0.59)",
    "suboptimal (< 0.4)",
    "hallucinatory",
    "missing",
]


# ═══════════════════════════════════════════════════════════════════
# LOADERS
# ═══════════════════════════════════════════════════════════════════

def load_roi_names(path):

    with open(path) as f:
        return json.load(f)


def load_brain_graph_texts(path):

    """
    Expected format:

    {
        "subject_id":
            "Node feature: ... Edge feature: ..."
    }
    """

    with open(path) as f:
        return json.load(f)


def load_network_names(path):

    with open(path) as f:
        return json.load(f)


def load_ground_truth_hub_map(path):

    """
    hub_ground_truth_20roi.json:

    {
        subject_id: [
            {
                "roi": "...",
                "degree": ...,
                "cohort_z": ...
            },
            ...
        ]
    }

    Returns:

    {
        subject_id: [
            roi_name,
            roi_name,
            ...
        ]
    }
    """

    with open(path) as f:
        raw = json.load(f)

    result = {}

    for sid, entries in raw.items():

        result[sid] = [
            entry["roi"]
            for entry in entries
        ]

    return result


def load_ground_truth_community_map(path, top_n):

    """
    Combines:

        within_network
        between_network

    into one ranked list.

    Returns:

    {
        subject_id: [
            "Network-Network",
            ...
        ]
    }
    """

    with open(path) as f:
        raw = json.load(f)

    result = {}

    for sid, record in raw.items():

        combined = {}

        # ----------------------------------------------------------
        # Within-network pairs
        # ----------------------------------------------------------

        for net, cnt in record["within_network"].items():

            combined[f"{net}-{net}"] = cnt

        # ----------------------------------------------------------
        # Between-network pairs
        # ----------------------------------------------------------

        for pair, cnt in record["between_network"].items():

            combined[pair] = cnt

        # ----------------------------------------------------------
        # Sort by count
        # ----------------------------------------------------------

        ranked = [
            pair
            for pair, count in sorted(
                combined.items(),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        result[sid] = ranked[:top_n]

    return result


# ═══════════════════════════════════════════════════════════════════
# VALID PAIRS
# ═══════════════════════════════════════════════════════════════════

def build_all_valid_pairs(network_names):

    """
    Creates all valid network pairs in canonical order.

    Example:

        NetworkA-NetworkA
        NetworkA-NetworkB
        NetworkA-NetworkC
        NetworkB-NetworkB
        ...
    """

    within_pairs = {
        f"{n}-{n}"
        for n in network_names
    }

    between_pairs = {
        f"{a}-{b}"
        for i, a in enumerate(network_names)
        for b in network_names[i + 1:]
    }

    return within_pairs | between_pairs


# ═══════════════════════════════════════════════════════════════════
# EDGE REGION MAP
# ═══════════════════════════════════════════════════════════════════

def build_edge_regions_map(brain_graph_texts):

    """
    Extract all tokens from each subject's graph text.

    Used to check whether a predicted ROI actually appears
    in the subject's graph.
    """

    return {
        sid: set(
            re.findall(
                r"[A-Za-z0-9_]+",
                text
            )
        )
        for sid, text in brain_graph_texts.items()
    }


# ═══════════════════════════════════════════════════════════════════
# NORMALIZE COMMUNITY PAIR ORDER
# ═══════════════════════════════════════════════════════════════════

def normalize_pair(pair, network_order):

    """
    Converts:

        Cont-DorsAttn

    into:

        DorsAttn-Cont

    if DorsAttn appears before Cont in network_order.
    """

    parts = pair.split("-")

    if len(parts) != 2:
        return pair

    a, b = parts

    if a not in network_order:
        return pair

    if b not in network_order:
        return pair

    index_a = network_order.index(a)
    index_b = network_order.index(b)

    if index_a <= index_b:

        return f"{a}-{b}"

    else:

        return f"{b}-{a}"


# ═══════════════════════════════════════════════════════════════════
# OVERLAP GRADING
# ═══════════════════════════════════════════════════════════════════

def grade_overlap(overlap_ratio):

    if overlap_ratio >= 0.6:

        return "correct (>= 0.6)"

    elif overlap_ratio >= 0.4:

        return "partial (0.4-0.59)"

    else:

        return "suboptimal (< 0.4)"


# ═══════════════════════════════════════════════════════════════════
# SCORE HUB RESPONSE
# ═══════════════════════════════════════════════════════════════════

def score_hub_response(
    predicted_hubs,
    ground_truth_top20,
    all_valid_regions,
    edge_regions
):

    # --------------------------------------------------------------
    # No prediction
    # --------------------------------------------------------------

    if not predicted_hubs:

        return "missing"

    # --------------------------------------------------------------
    # Validate every predicted ROI
    # --------------------------------------------------------------

    for region in predicted_hubs:

        # Not an actual ROI
        if (
            all_valid_regions
            and region not in all_valid_regions
        ):

            return "hallucinatory"

        # ROI does not appear in subject graph
        if (
            edge_regions
            and region not in edge_regions
        ):

            return "hallucinatory"

    # --------------------------------------------------------------
    # Calculate overlap
    # --------------------------------------------------------------

    overlap = len(
        set(predicted_hubs)
        &
        set(ground_truth_top20)
    )

    overlap_ratio = (
        overlap /
        max(len(predicted_hubs), 1)
    )

    return grade_overlap(overlap_ratio)


# ═══════════════════════════════════════════════════════════════════
# SCORE COMMUNITY RESPONSE
# ═══════════════════════════════════════════════════════════════════

def score_community_response(
    predicted_pairs,
    gt_top10,
    all_valid_pairs,
    network_names
):

    # --------------------------------------------------------------
    # No prediction
    # --------------------------------------------------------------

    if not predicted_pairs:

        return "missing"

    # --------------------------------------------------------------
    # Normalize pair ordering
    # --------------------------------------------------------------

    predicted_pairs = [
        normalize_pair(
            p,
            network_names
        )
        for p in predicted_pairs
    ]

    # --------------------------------------------------------------
    # Validate pairs
    # --------------------------------------------------------------

    for pair in predicted_pairs:

        if pair not in all_valid_pairs:

            return "hallucinatory"

    # --------------------------------------------------------------
    # Calculate overlap
    # --------------------------------------------------------------

    matches = [
        p
        for p in predicted_pairs
        if p in gt_top10
    ]

    overlap_ratio = (
        len(matches) /
        max(len(predicted_pairs), 1)
    )

    return grade_overlap(overlap_ratio)


# ═══════════════════════════════════════════════════════════════════
# IDENTIFY MODEL / TASK / CONDITION
# ═══════════════════════════════════════════════════════════════════

def parse_run_info(path, rq_root):

    """
    Uses directory structure:

        RQ_4_5/
            MODEL/
                TASK/
                    FILE.json

    Example:

        RQ_4_5/deepseek/hub/
        deepseek_hub_for_classification_no_edge_20hub.json

    Returns:

        model = deepseek
        task = hub
        condition = no_edge
    """

    relative_path = os.path.relpath(
        path,
        rq_root
    )

    parts = relative_path.split(
        os.sep
    )

    # Need:
    #
    # model / task / filename

    if len(parts) < 3:

        return None

    model = parts[0]

    task = parts[1]

    filename = parts[-1].lower()

    # --------------------------------------------------------------
    # Validate task
    # --------------------------------------------------------------

    if task not in {
        "hub",
        "community"
    }:

        return None

    # --------------------------------------------------------------
    # Determine condition
    # --------------------------------------------------------------

    if "no_edge" in filename:

        condition = "no_edge"

    elif "no_weight" in filename:

        condition = "no_weight"

    elif "edge_no_weight" in filename:

        condition = "no_weight"

    else:

        condition = "full"

    return model, task, condition


# ═══════════════════════════════════════════════════════════════════
# FIND TARGET FILES
# ═══════════════════════════════════════════════════════════════════

def find_target_files():

    """
    Recursively searches:

        /home/siu856622573/Code/RQ_4_5

    for the 12 requested JSON files.
    """

    all_files = glob.glob(
        os.path.join(
            RQ45_DIR,
            "**",
            "*.json"
        ),
        recursive=True
    )

    selected_files = []

    for path in all_files:

        filename = os.path.basename(path)

        if filename in TARGET_FILES:

            selected_files.append(path)

    return sorted(selected_files)


# ═══════════════════════════════════════════════════════════════════
# HUB BUCKET COUNTS
# ═══════════════════════════════════════════════════════════════════

def hub_bucket_counts(
    data,
    ground_truth_hub_map,
    all_valid_regions,
    edge_regions_map
):

    counts = {
        bucket: 0
        for bucket in BUCKETS
    }

    for sid, record in data.items():

        predicted = (
            record.get("hub_regions")
            or []
        )

        gt_top20 = (
            ground_truth_hub_map.get(
                sid,
                []
            )
        )

        edge_regions = (
            edge_regions_map.get(
                sid,
                set()
            )
        )

        label = score_hub_response(
            predicted,
            gt_top20,
            all_valid_regions,
            edge_regions
        )

        counts[label] += 1

    return counts


# ═══════════════════════════════════════════════════════════════════
# COMMUNITY BUCKET COUNTS
# ═══════════════════════════════════════════════════════════════════

def community_bucket_counts(
    data,
    ground_truth_community_map,
    all_valid_pairs,
    network_names
):

    counts = {
        bucket: 0
        for bucket in BUCKETS
    }

    for sid, record in data.items():

        predicted = (
            record.get("community_pairs")
            or []
        )

        gt_top10 = (
            ground_truth_community_map.get(
                sid,
                []
            )
        )

        label = score_community_response(
            predicted,
            gt_top10,
            all_valid_pairs,
            network_names
        )

        counts[label] += 1

    return counts


# ═══════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════

def build_bucket_breakdown():

    print()
    print("=" * 90)
    print("LOADING GROUND TRUTH / RESOURCE FILES")
    print("=" * 90)

    # --------------------------------------------------------------
    # Load ROI names
    # --------------------------------------------------------------

    roi_names = load_roi_names(
        ROI_NAMES_JSON
    )

    # --------------------------------------------------------------
    # Load graph texts
    # --------------------------------------------------------------

    brain_graph_texts = load_brain_graph_texts(
        EDGE_LIST_NAMES_ONLY
    )

    edge_regions_map = build_edge_regions_map(
        brain_graph_texts
    )

    # --------------------------------------------------------------
    # Load network names
    # --------------------------------------------------------------

    network_names = load_network_names(
        NETWORK_NAMES_JSON
    )

    # --------------------------------------------------------------
    # Load hub ground truth
    # --------------------------------------------------------------

    ground_truth_hub_map = (
        load_ground_truth_hub_map(
            HUB_GROUND_TRUTH
        )
    )

    # --------------------------------------------------------------
    # Load community ground truth
    # --------------------------------------------------------------

    ground_truth_community_map = (
        load_ground_truth_community_map(
            COMMUNITY_GROUND_TRUTH_COUNTS,
            TOP_K_GROUND_TRUTH
        )
    )

    # --------------------------------------------------------------
    # Build valid sets
    # --------------------------------------------------------------

    all_valid_regions = set(
        roi_names
    )

    all_valid_pairs = (
        build_all_valid_pairs(
            network_names
        )
    )

    print(
        f"Number of valid ROI names: "
        f"{len(all_valid_regions)}"
    )

    print(
        f"Number of valid network pairs: "
        f"{len(all_valid_pairs)}"
    )

    # ══════════════════════════════════════════════════════════════
    # FIND FILES
    # ══════════════════════════════════════════════════════════════

    print()
    print("=" * 90)
    print("SEARCHING FOR TARGET JSON FILES")
    print("=" * 90)

    selected_files = find_target_files()

    print(
        f"\nFound {len(selected_files)} "
        f"of {len(TARGET_FILES)} target files.\n"
    )

    for path in selected_files:

        print(
            "FOUND:",
            path
        )

    # --------------------------------------------------------------
    # Report missing files
    # --------------------------------------------------------------

    found_names = {
        os.path.basename(path)
        for path in selected_files
    }

    missing_files = (
        TARGET_FILES -
        found_names
    )

    if missing_files:

        print()
        print("WARNING: Missing target files:")

        for filename in sorted(
            missing_files
        ):

            print(
                "  MISSING:",
                filename
            )

    # --------------------------------------------------------------
    # Stop if nothing found
    # --------------------------------------------------------------

    if not selected_files:

        print()
        print(
            "ERROR: No target JSON files were found."
        )

        return None

    # ══════════════════════════════════════════════════════════════
    # PROCESS FILES
    # ══════════════════════════════════════════════════════════════

    rows = []

    print()
    print("=" * 90)
    print("PROCESSING FILES")
    print("=" * 90)

    for path in selected_files:

        filename = os.path.basename(
            path
        )

        # ----------------------------------------------------------
        # Determine model / task / condition
        # ----------------------------------------------------------

        info = parse_run_info(
            path,
            RQ45_DIR
        )

        if info is None:

            print(
                f"\nWARNING: Could not parse:\n"
                f"{path}"
            )

            continue

        model, task, condition = info

        # ----------------------------------------------------------
        # Print file information
        # ----------------------------------------------------------

        print()
        print("-" * 90)

        print(
            f"Model     : {model}"
        )

        print(
            f"Task      : {task}"
        )

        print(
            f"Condition : {condition}"
        )

        print(
            f"File      : {filename}"
        )

        print(
            f"Path      : {path}"
        )

        # ----------------------------------------------------------
        # Load JSON
        # ----------------------------------------------------------

        try:

            with open(path) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"ERROR loading {path}: {e}"
            )

            continue

        print(
            f"N         : {len(data)}"
        )

        # ----------------------------------------------------------
        # Score
        # ----------------------------------------------------------

        if task == "hub":

            counts = hub_bucket_counts(
                data,
                ground_truth_hub_map,
                all_valid_regions,
                edge_regions_map
            )

        elif task == "community":

            counts = community_bucket_counts(
                data,
                ground_truth_community_map,
                all_valid_pairs,
                network_names
            )

        else:

            continue

        # ----------------------------------------------------------
        # Print bucket results
        # ----------------------------------------------------------

        print()
        print("Bucket counts:")

        for bucket in BUCKETS:

            print(
                f"  {bucket:<25} "
                f"{counts[bucket]}"
            )

        # ----------------------------------------------------------
        # Create dataframe row
        # ----------------------------------------------------------

        row = {
            "Model": model,
            "Task": task,
            "Condition": condition,
            "N": len(data),
            "File": filename,
        }

        row.update(
            counts
        )

        rows.append(
            row
        )

    # ══════════════════════════════════════════════════════════════
    # CREATE DATAFRAME
    # ══════════════════════════════════════════════════════════════

    if not rows:

        print()
        print(
            "ERROR: No JSON files were successfully processed."
        )

        return None

    df = pd.DataFrame(
        rows
    )

    df = df.sort_values(
        [
            "Task",
            "Model",
            "Condition"
        ]
    ).reset_index(
        drop=True
    )

    # ══════════════════════════════════════════════════════════════
    # COMMUNITY RESULTS
    # ══════════════════════════════════════════════════════════════

    print()
    print()
    print("=" * 120)
    print("COMMUNITY BUCKET COUNTS")
    print("=" * 120)

    community_columns = [
        "Model",
        "Condition",
        "N",
    ] + BUCKETS + [
        "File"
    ]

    comm_df = df[
        df["Task"] == "community"
    ][community_columns]

    if len(comm_df) > 0:

        print(
            comm_df.to_string(
                index=False
            )
        )

    else:

        print(
            "No community files processed."
        )

    # ══════════════════════════════════════════════════════════════
    # HUB RESULTS
    # ══════════════════════════════════════════════════════════════

    print()
    print()
    print("=" * 120)
    print("HUB BUCKET COUNTS")
    print("=" * 120)

    hub_columns = [
        "Model",
        "Condition",
        "N",
    ] + BUCKETS + [
        "File"
    ]

    hub_df = df[
        df["Task"] == "hub"
    ][hub_columns]

    if len(hub_df) > 0:

        print(
            hub_df.to_string(
                index=False
            )
        )

    else:

        print(
            "No hub files processed."
        )

    # ══════════════════════════════════════════════════════════════
    # SAVE CSV
    # ══════════════════════════════════════════════════════════════

    df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print()
    print()
    print("=" * 90)
    print("DONE")
    print("=" * 90)

    print(
        f"\nSaved bucket breakdown to:"
    )

    print(
        OUTPUT_CSV
    )

    print()

    return df


# ═══════════════════════════════════════════════════════════════════
# RUN
# ═══════════════════════════════════════════════════════════════════

bucket_df = build_bucket_breakdown()


LOADING GROUND TRUTH / RESOURCE FILES
Number of valid ROI names: 116
Number of valid network pairs: 36

SEARCHING FOR TARGET JSON FILES

Found 12 of 12 target files.

FOUND: /home/siu856622573/Code/RQ_4_5/deepseek/community/deepseek_community_for_classification_no_edge_10com.json
FOUND: /home/siu856622573/Code/RQ_4_5/deepseek/hub/deepseek_hub_for_classification_no_edge_20hub.json
FOUND: /home/siu856622573/Code/RQ_4_5/deepseek_v4/community/deepseek_v4_community_for_classification_10com.json
FOUND: /home/siu856622573/Code/RQ_4_5/deepseek_v4/hub/deepseek_v4_hub_for_classification_20hub.json
FOUND: /home/siu856622573/Code/RQ_4_5/gpt_luna/community/gpt_community_for_classification_no_edge_10com.json
FOUND: /home/siu856622573/Code/RQ_4_5/gpt_luna/hub/gpt_hub_for_classification_20hub.json
FOUND: /home/siu856622573/Code/RQ_4_5/llama/community/llama_community_for_classification_10com.json
FOUND: /home/siu856622573/Code/RQ_4_5/llama/hub/llama_hub_for_classification_20hub_del.json
FOUND: /home/s

## Peek at the results

In [4]:
print(bucket_df[bucket_df["Task"] == "community"].head(3))
print(bucket_df[bucket_df["Task"] == "hub"].head(3))

with open(HUB_GROUND_TRUTH) as f:
    sample = json.load(f)
first_sid = next(iter(sample))
print(first_sid, sample[first_sid][:3])


      Model       Task  Condition    N  \
0  deepseek  community       full  768   
1  deepseek  community    no_edge  768   
2  deepseek  community  no_weight  768   

                                                File  correct (>= 0.6)  \
0   deepseek_community_for_classification_10com.json                12   
1  deepseek_community_for_classification_no_edge_...                15   
2  deepseek_community_for_classification_no_edge_...                14   

   partial (0.4-0.59)  suboptimal (< 0.4)  hallucinatory  missing  
0                  51                 512              1      192  
1                  51                 515              3      184  
2                  52                 515              3      184  
       Model Task  Condition    N  \
19  deepseek  hub       full  768   
20  deepseek  hub    no_edge  768   
21  deepseek  hub  no_weight  768   

                                                 File  correct (>= 0.6)  \
19         deepseek_hub_for_classifica